In [ ]:
from parsers.lr.lr1 import LR1Parser
from parsers.lr.lr_parsing_engine import LREngine

In [60]:
grammar = [
    "P -> L $",
    "S -> id := id",
    "S -> if id then S",
    "S -> if id then S else S",
    "L -> S",
    "L -> L ; S",
]

In [61]:
p = LR1Parser(*grammar).to_lalr()

In [62]:
print(p)

S─┬─ if id then S else S
  ├─ id := id
  └─ if id then S
L─┬─ L ; S
  └─ S
P─── L $



In [63]:
print(p.to_tabulate())

┌────┬─────┬──────┬──────┬─────┬──────┬────────┬────────┬─────┬─────┬─────┐
│    │ $   │ id   │ :=   │ ;   │ if   │ else   │ then   │ L   │ S   │ P   │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  0 │ r5  │      │      │ r5  │      │ s6, r5 │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  1 │     │ s5   │      │     │      │        │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  3 │ r0  │      │      │ r0  │      │ r0     │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  4 │     │      │ s1   │     │      │        │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  5 │ r1  │      │      │ r1  │      │ r1     │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  6 │     │

In [64]:
cell = list(p.parsing_table[0]["else"])
cell[:1]
p.parsing_table[0]["else"] = set(cell[:1])

In [65]:
print(p.to_tabulate())

┌────┬─────┬──────┬──────┬─────┬──────┬────────┬────────┬─────┬─────┬─────┐
│    │ $   │ id   │ :=   │ ;   │ if   │ else   │ then   │ L   │ S   │ P   │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  0 │ r5  │      │      │ r5  │      │ s6     │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  1 │     │ s5   │      │     │      │        │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  3 │ r0  │      │      │ r0  │      │ r0     │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  4 │     │      │ s1   │     │      │        │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  5 │ r1  │      │      │ r1  │      │ r1     │        │     │     │     │
├────┼─────┼──────┼──────┼─────┼──────┼────────┼────────┼─────┼─────┼─────┤
│  6 │     │

In [66]:
p.print_rules_and_states()

---=== Rules ===---
0000 S -> if id then S else S
0001 S -> id := id
0002 L -> L ; S
0003 L -> S
0004 P -> L $
0005 S -> if id then S

---=== States ===---
state 0
    S -> if id then S . ($ ; else)
    S -> if id then S . else S ($ ; else)

state 1
    S -> id := . id ($ ; else)

state 3
    S -> if id then S else S . ($ ; else)

state 4
    S -> id . := id ($ ; else)

state 5
    S -> id := id . ($ ; else)

state 6
    S -> . id := id ($ ; else)
    S -> . if id then S ($ ; else)
    S -> if id then S else . S ($ ; else)
    S -> . if id then S else S ($ ; else)

state 7
    S -> if id then . S ($ ; else)
    S -> . id := id ($ ; else)
    S -> . if id then S ($ ; else)
    S -> if id then . S else S ($ ; else)
    S -> . if id then S else S ($ ; else)

state 8
    L -> S . ($ ;)

state 11
    L -> L . ; S ($ ;)
    P -> L . $

state 13
    S -> if . id then S else S ($ ; else)
    S -> if . id then S ($ ; else)

state 14
    S -> if id . then S ($ ;)
    S -> if id . then S else S (

In [67]:
e = LREngine(p)

In [68]:
code_sample = """
if id then     
    if id then
        id := id
    else
        id := id ;
        id := id $
""".split()
code_sample

['if',
 'id',
 'then',
 'if',
 'id',
 'then',
 'id',
 ':=',
 'id',
 'else',
 'id',
 ':=',
 'id',
 ';',
 'id',
 ':=',
 'id',
 '$']

In [69]:
def print_iter(stack, states, symbol, action):
    for el in stack:
        print_el = next(iter(el.keys())) if isinstance(el, dict) else el
        print(print_el, end=", ")
    print(f" {symbol=}, {states[-1]=}, {action=}")

In [70]:
import json

In [71]:
out = e.parse(code_sample, iteration_callback=print_iter)
print(json.dumps(out, indent=2))

 symbol='if', states[-1]=22, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=13)
if,  symbol='id', states[-1]=13, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=14)
if, id,  symbol='then', states[-1]=14, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=7)
if, id, then,  symbol='if', states[-1]=7, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=13)
if, id, then, if,  symbol='id', states[-1]=13, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=14)
if, id, then, if, id,  symbol='then', states[-1]=14, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=7)
if, id, then, if, id, then,  symbol='id', states[-1]=7, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=4)
if, id, then, if, id, then, id,  symbol=':=', states[-1]=4, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=1)
if, id, then, if, id, then, id, :=,  symbol='id', states[-1]=1, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=5)
if, id, then, if, id, then, id, :=, id,  symbol='else', states[-1]=5, action=L

In [14]:
p.get_starting_state_idx()

22